# Supervised Learning. Classification: Class imbalance


📖 **Class imbalance** is a machine learning challenge where a dataset has a disproportionate number of examples across its classes, with one or more classes having significantly fewer samples than the others. This leads to models that are biased towards the majority class and perform poorly on the minority class, often ignoring it completely. Addressing it is crucial for reliable model performance and can involve techniques like oversampling the minority class, undersampling the majority class, or using algorithms that handle imbalance directly. 

**What's this Notebook about?** We aim to build a robust classification model to predict the presence of diabetes, facing the challenge of a highly imbalanced dataset where positive cases (diabetes) are significantly lower than negative ones (non-diabetes). We will employ specific strategies to ensure the model learns to identify the minority class effectively, preventing it from becoming biased toward the majority baseline.


* In this notebook, we will utilize the ``RandomForestClassifier``. The objective is to demonstrate how to handle imbalanced datasets and to evaluate and compare the performance results obtained by applying various class balancing techniques

### Load libraries

In [ ]:
import pandas as pd

import warnings
warnings.filterwarnings('ignore')

## Load the dataset

Dataset can be found in [Kaggle](#https://www.kaggle.com/datasets/alexteboul/diabetes-health-indicators-dataset) 

The target variable **Diabetes_binary** has 2 classes. 
* 0 is for no diabetes, 
* 1 is for prediabetes or diabetes. 


This dataset has 21 feature variables and is not balanced.


In [ ]:
data = pd.read_csv('Data/S4_diabetes_desequilibrio_clases.csv')
data.head()

How many data we have?

In [ ]:
data.shape

Let's check more info about the dataset

In [ ]:
data.info()

### What represents each column?

**(LABEL) Diabetes_binary:** Description: Indicates whether an individual has diabetes (1) or not (0). This is likely the target or outcome variable of the dataset.


**HighBP:** Indicates if the person has high blood pressure (1) or not (0).


**HighChol:** Indicates if the individual has high cholesterol levels (1) or not (0).

**CholCheck:** Whether the individual has had their cholesterol checked (1) in the last 5 years or not (0).

**BMI:** The individual’s Body Mass Index (BMI), a measure of body fat based on height and weight.

**Smoker:** Indicates whether the individual is a smoker (1) or not (0).


**Stroke:** Indicates if the individual has had a stroke (1) or not (0).


**HeartDiseaseorAttack:** Whether the individual has had coronary heart disease or myocardial infarction (1) or not (0).

**PhysActivity:** Indicates if the individual has engaged in physical activity or exercise in the last 30 days - not including job (1) or not (0).


**Fruits:** Indicates whether the individual consumes 1 fruit or more daily (1) or not (0).


**Veggies:** Indicates whether the individual consumes 1 vegetable or more daily (1) or not (0).

**HeavyAlcoholCons:** Heavy drinkers (adult men having more than 14 drinks per week and adult women having more than 7 drinks per week) (1) or not (0)

**AnyHealthcare:** Indicates whether the person has access to any form of healthcare coverage (1) or not (0).

**NoDocbcCost:** Whether the person could not visit a doctor in the past year due to cost (1) or not (0).

**GenHlth:** Self-reported general health status, where lower values (1) indicate better health (e.g., excellent) and higher values (5) indicate worse health (e.g., poor).

**MentHlth:** Number of days in the past 30 days that the individual experienced poor mental health (stress, depression, and problems with emotions).

**PhysHlth:** Number of days in the past 30 days that the individual experienced poor physical health (physical illness and injury).

**DiffWalk:** Indicates whether the individual has difficulty walking or climbing stairs (1) or not (0).


**Sex:** Gender of the individual, 0 = female; 1 = male.

**Age:** Represents age group categories. 13-level age category. 
* 1: 18-24 years old
* 2: 25-29 years old
* ...
* 13: 80 years or older

**Education:** Represents the highest level of education attained. 
* 1: Never attended school
* 2: Elementary school
* ...
* 6: College graduate.

**Income:** Represents income categories. Scale 1-8

* 1: Less than 10 000 
* 2: 10 000 to 14 999
* ...
* 8: 75 000 or more.

Show some statistic summary of the data

In [ ]:
data.describe()

### Since this is a binary classification problem, we need to know if there is a class imbalance.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

print("Class 0:", (data['Diabetes_binary'] == 0).sum())
print("Class 1:", (data['Diabetes_binary'] == 1).sum())



<div style="background-color:#ccffcc; padding:10px; border-radius:5px;">

### <span style="color:blue">Exercise 1</span>
What's the percentage of the minor class?
    </div>

In [ ]:

# write your code here 

percentage_minor_class = data['Diabetes_binary'].value_counts().values[1]/len(data)
percentage_minor_class



Let's separate the data into train and test. Then, train the models using accuracy and balance accuracy metrics. Then, comapre results. 

In [ ]:
# Separate from train and test
from sklearn.model_selection import train_test_split

# Split the data FIRST
X = data.drop("Diabetes_binary", axis=1)
y = data["Diabetes_binary"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


All the results from the different iterations will be save in a dataframe

In [ ]:
import pandas as pd

results_df = pd.DataFrame(columns=["Model", "Accuracy", "Balanced Accuracy", "Recall", "Precision"])

## With imbalance classes, we need to check more Evaluation Metrics.

As we have seen above, accuracy is not the best metric for evaluating unbalanced data sets, as it can be misleading. Metrics that can provide better insight include:

**Confusion matrix**: a table showing correct predictions and types of incorrect predictions.

* **Accuracy**: the number of true positives divided by all positive predictions. Precision is also called positive predictive value. It is a measure of the accuracy of a classifier. A low precision indicates a high number of false positives.

* **Recall**: the number of true positives divided by the number of positive values in the test data, also called sensitivity or true positive rate. It is a measure of the completeness of a classifier. A low recall indicates a high number of false negatives.

* **F1 score**: the weighted average of precision and recall.




**Precision for Class 1 (Diabetes):**
* Precision tells us how many of the instances predicted as diabetes (class 1) are actually diabetic.
* For isntance, if precision for class 1 is 0.42, this means that when the model predicts someone has diabetes, it is correct 42% of the time. The remaining 58% of the predictions are false positives (people without diabetes incorrectly predicted as having diabetes).

**Recall for Class 1 (Diabetes):**
* Recall (also known as sensitivity or true positive rate) measures how many of the actual diabetic cases were correctly identified by the model.
* Recall for class 1 is 0.17, meaning that the model correctly identifies only 17% of the people who actually have diabetes.

**F1-Score for Class 1 (Diabetes):**
* The F1-score is the harmonic mean of precision and recall, combining both into a single metric. It balances the two metrics, making it useful when the data is imbalanced, as is the case here.


### Create a function to plot the classification metrics

In [ ]:
from sklearn.metrics import confusion_matrix

# Function to plot confusion matrix
def plot_confusion_matrix(y_test, y_pred, model_name):
    cm = confusion_matrix(y_pred, y_test)
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap=sns.cubehelix_palette(as_cmap=True), cbar=False)
    plt.title(f"Confusion Matrix - {model_name}")
    plt.ylabel('Predicted Label')
    plt.xlabel('Actual Label')
    plt.show()

## First, calculate the classifier baseline. 

LogisticRegression (linear model) and Random Forest are selected. 

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix



log_reg = LogisticRegression(random_state=42)
log_reg.fit(X_train, y_train)  #train the model

y_pred_logreg_baseline = log_reg.predict(X_test)  # Make predictions

# Evaluation metrics
acc_lg =  accuracy_score(y_test, y_pred_logreg_baseline)
balanced_acc_lg = balanced_accuracy_score(y_test, y_pred_logreg_baseline)

recall_lg = recall_score(y_test, y_pred_logreg_baseline, average='binary')
precision_lg = precision_score(y_test, y_pred_logreg_baseline, average='binary')


In [ ]:
# Save the results in the Dataframe

results_df.loc[len(results_df)] = [
    "Logistic Regression (Baseline)",
    acc_lg,
    balanced_acc_lg,
    recall_lg,
    precision_lg
]

In [ ]:

plot_confusion_matrix(y_test, y_pred_logreg_baseline, "LogisticRegression")


In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Random Forest
random_forest = RandomForestClassifier(random_state=42)
random_forest.fit(X_train, y_train)

# Predictions
y_pred_forest_baseline = random_forest.predict(X_test)


acc_rf = accuracy_score(y_test, y_pred_forest_baseline)
balanced_acc_rf = balanced_accuracy_score(y_test, y_pred_forest_baseline)

recall_rf = recall_score(y_test, y_pred_forest_baseline, average='binary')
precision_rf = precision_score(y_test, y_pred_forest_baseline, average='binary')


In [ ]:
# Save the results in the Dataframe

results_df.loc[len(results_df)] = [
    "Random Forest (Baseline)",
    acc_rf,
    balanced_acc_rf, 
    recall_rf,
    precision_rf
]

In [ ]:

plot_confusion_matrix(y_test, y_pred_forest_baseline, "RandomForest")


In [ ]:
results_df

It is evident that there is a class imbalance problem. Now, we will study the methods to address this problem.

# **Methods to handle imbalance datasets**: 
* Resampling the original dataset: 
    * Oversampling
    * Undersampling
    
* Oversampling with synthetic data (SMOTE). Instead of replicating, we are creating new instances.
* Using class weight in model training
* Change classification evaluation metric to identify class imbalance


📖 [imbalanced-learn documentation](#https://imbalanced-learn.org/stable/ ) Imbalanced-learn (imported as imblearn) is an open source, MIT-licensed library relying on scikit-learn (imported as sklearn) and provides tools when dealing with classification with imbalanced classes.


**We will see first the code to implementing this techniques. Then, different models will be tested.**

## Resampling the original dataset


`sklearn.utils.resample(*arrays, replace=True, n_samples=None, random_state=None, stratify=None)`

[Scikit learn documentation](#https://scikit-learn.org/1.5/modules/generated/sklearn.utils.resample.html)

### Undersampling vs oversampling

A widely adopted technique for dealing with highly imbalanced datasets is called resampling. It involves removing samples from the majority class (undersampling) and/or adding more examples from the minority class (oversampling).

<img src="Figures/resampling.png" alt="Drawing" style="width: 800px;"/>



### You must always separate the training and testing sets before you apply oversampling or generating synthetic data.

If you oversample the entire dataset first and then split it, you will cause Data Leakage. This is one of the most common pitfalls in imbalanced classification.

* **Why is this critical?** The goal of the ``test set`` is to simulate "unseen" real-world data.

* If you oversample before splitting: You create synthetic copies (or exact duplicates) of the minority class samples. Some of these copies will end up in the ``training set``. Your model will simply "memorize" these examples rather than learning to generalize. **You will get incredibly high accuracy scores (>95%) during testing, but the model will fail completely in production**.

* If you oversample after splitting: You split the data first. Then, you only inflate the minority class in the ``training set``. The ``test set`` remains untouched and maintains the original, realistic imbalance. **This ensures your evaluation metrics are honest**.

# 1. Option 1) Oversampling the original dataset. 


In [ ]:
# Separate from train and test
from sklearn.model_selection import train_test_split

# Split the data FIRST
X = data.drop("Diabetes_binary", axis=1)
y = data["Diabetes_binary"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Recombine ``X_train`` and ``y_train`` into a single DataFrame. This is necessary because some resample methods need to filter by the class label.

### **1.1 Random Oversampling the minority class**

Oversampling can be defined as adding more copies of the minority class. Oversampling can be a good option when there isn't a lot of data to work with.

We will use the resampling module from Scikit-Learn to randomly replicate samples from the minority class.

In [ ]:
# if you have not installed imblearn, run the line below
# !pip install imblearn

In [ ]:
from imblearn.over_sampling import RandomOverSampler

ros = RandomOverSampler(random_state=42)
X_train_over, y_train_over = ros.fit_resample(X_train, y_train)


In [ ]:
X_train_over

###  How many classes of each do we have now?


In [ ]:
print("Class 0:", (y_train_over == 0).sum())
print("Class 1:", (y_train_over == 1).sum())

### Now, let's train the model with the Oversample training dataset

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score


log_reg = LogisticRegression(random_state=42)
log_reg.fit(X_train_over, y_train_over)  #train the model

y_pred_logreg_over = log_reg.predict(X_test)  # Make predictions

# Evaluation metrics
acc_lg =  accuracy_score(y_test, y_pred_logreg_over)
balanced_acc_lg = balanced_accuracy_score(y_test, y_pred_logreg_over)


recall_lg = recall_score(y_test, y_pred_logreg_over, average='binary')
precision_lg = precision_score(y_test, y_pred_logreg_over, average='binary')


In [ ]:
# Save the results in the Dataframe

results_df.loc[len(results_df)] = [
    "Logistic Regression (Oversampling)",
    acc_lg,
    balanced_acc_lg,
    recall_lg,
    precision_lg
]

In [ ]:
plot_confusion_matrix(y_test, y_pred_logreg_over, "LogisticRegression")
print(classification_report(y_test, y_pred_logreg_over))

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Random Forest
random_forest = RandomForestClassifier(random_state=42)
random_forest.fit(X_train_over, y_train_over)

# Predictions
y_pred_forest_over = random_forest.predict(X_test)

# Evaluation metrics
acc_rf = accuracy_score(y_test, y_pred_forest_over)
balanced_acc_rf = balanced_accuracy_score(y_test, y_pred_forest_over)

recall_rf = recall_score(y_test, y_pred_forest_over, average='binary')
precision_rf = precision_score(y_test, y_pred_forest_over, average='binary')

In [ ]:
# Save the results in the Dataframe

results_df.loc[len(results_df)] = [
    "Random Forest (Oversampling)",
    acc_rf,
    balanced_acc_rf,
    recall_rf,
    precision_rf
]

In [ ]:
plot_confusion_matrix(y_test, y_pred_forest_over, "RandomForest")
print(classification_report(y_test, y_pred_forest_over))

In [ ]:
results_df

### Create a function for plotting the confusion matrix during the exercise

# **Option 2) Undersampling the original dataset Random  Undersampling the Majority Class**

Undersampling can be defined as the removal of some observations from the majority class. Undersampling can be a good option when there is a large amount of data available, for example, millions of rows. However, the downside is that we are eliminating information that could be valuable. This could lead to underfitting and poor generalization on the test set.


In [ ]:
from imblearn.under_sampling import RandomUnderSampler

rus = RandomUnderSampler(random_state=42)

X_train_under, y_train_under = rus.fit_resample(X_train, y_train)


How many classes do we have now? 

In [ ]:
print("Class 0:", (y_train_under == 0).sum())
print("Class 1:", (y_train_under == 1).sum())

### Now, let's train the model with the Undersampled training dataset

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score


log_reg = LogisticRegression(random_state=42)
log_reg.fit(X_train_under, y_train_under)  #train the model

y_pred_logreg_under = log_reg.predict(X_test)  # Make predictions

# Evaluation metrics
acc_lg =  accuracy_score(y_test, y_pred_logreg_under)
balanced_acc_lg = balanced_accuracy_score(y_test, y_pred_logreg_under)

recall_lg = recall_score(y_test, y_pred_logreg_under, average='binary')
precision_lg = precision_score(y_test, y_pred_logreg_under, average='binary')

In [ ]:
# Save the results in the Dataframe

results_df.loc[len(results_df)] = [
    "Logistic Regression (Undersampling)",
    acc_lg,
    balanced_acc_lg,
    recall_lg,
    precision_lg
]

In [ ]:
plot_confusion_matrix(y_test, y_pred_forest_over, "LogisticRegression")
print(classification_report(y_test, y_pred_logreg_under))

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Random Forest
random_forest = RandomForestClassifier(random_state=42)
random_forest.fit(X_train_under, y_train_under)

# Predictions
y_pred_forest_under = random_forest.predict(X_test)

acc_rf = accuracy_score(y_test, y_pred_forest_under)
balanced_acc_rf = balanced_accuracy_score(y_test, y_pred_forest_under)

recall_rf = recall_score(y_test, y_pred_forest_under, average='binary')
precision_rf = precision_score(y_test, y_pred_forest_under, average='binary')

In [ ]:
# Save the results in the Dataframe

results_df.loc[len(results_df)] = [
    "Random Forest (Undersampling)",
    acc_rf,
    balanced_acc_rf,
    recall_rf,
    precision_rf
]

In [ ]:
plot_confusion_matrix(y_test, y_pred_forest_over, "RandomForest")
print(classification_report(y_test, y_pred_forest_under))

In [ ]:
results_df

## Option 3) SMOTE: Oversampling using synthetic data

Several more sophisticated resampling techniques have been proposed in the literature.

For example, in oversampling, instead of creating exact copies of records from the minority class, we can introduce small variations in those copies, creating more diverse synthetic samples.

We are going to apply this resampling technique (creation of synthetic data) using the **[imbalanced-learn](https://imbalanced-learn.org/stable/)** Python library. It is compatible with scikit-learn and is part of the scikit-learn-contrib projects.

### Over-sampling: SMOTE

**Synthetic Minority Oversampling Technique (SMOTE)** 

SMOTE is a more sophisticated technique than random oversampling. It generates synthetic samples for the minority class by interpolating between existing samples.


<img src="Figures/SMOTE.PNG" alt="Drawing" style="width: 600px"/>


In [ ]:
from imblearn.over_sampling import SMOTE

# Apply SMOTE to upsample the minority class
smote = SMOTE(random_state=27)  # Initializes the SMOTE oversampler with a random seed for reproducibility.
X_smote, y_smote = smote.fit_resample(X_train, y_train)  #  Applies SMOTE to balance the minority class by generating synthetic samples.

print("Class 0:", (y_smote == 0).sum())
print("Class 1:", (y_smote == 1).sum())

### Let's see how the models perform

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score


log_reg = LogisticRegression(random_state=42)
log_reg.fit(X_smote, y_smote)  #train the model

y_pred_logreg_smote = log_reg.predict(X_test)  # Make predictions

# Evaluation metrics
acc_lg =  accuracy_score(y_test, y_pred_logreg_smote)
balanced_acc_lg = balanced_accuracy_score(y_test, y_pred_logreg_smote)


recall_lg = recall_score(y_test, y_pred_logreg_smote, average='binary')
precision_lg = precision_score(y_test, y_pred_logreg_smote, average='binary')

In [ ]:
# Save the results in the Dataframe

results_df.loc[len(results_df)] = [
    "Logistic Regression (SMOTE)",
    acc_lg,
    balanced_acc_lg,
    recall_lg,
    precision_lg
]

In [ ]:
plot_confusion_matrix(y_test, y_pred_logreg_smote, "Logistic Regression")
print(classification_report(y_test, y_pred_logreg_smote))

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Random Forest
random_forest = RandomForestClassifier(random_state=42)
random_forest.fit(X_smote, y_smote)

# Predictions
y_pred_forest_smote = random_forest.predict(X_test)


acc_rf = accuracy_score(y_test, y_pred_forest_smote)
balanced_acc_rf = balanced_accuracy_score(y_test, y_pred_forest_smote)
recall_rf = recall_score(y_test, y_pred_forest_smote, average='binary')
precision_rf = precision_score(y_test, y_pred_forest_smote, average='binary')

In [ ]:
# Save the results in the Dataframe

results_df.loc[len(results_df)] = [
    "Random Forest (SMOTE)",
    acc_rf,
    balanced_acc_rf,
    recall_rf,
    precision_rf
]

In [ ]:
plot_confusion_matrix(y_test, y_pred_forest_smote, "Random Forest")
print(classification_report(y_test, y_pred_forest_smote))

In [ ]:
results_df

## Option 4) Algorithms with Class Weights

Most of the models in scikit-learn have a parameter ``class_weight``. This parameter will affect the computation of the loss in linear model or the criterion in the tree-based model to penalize differently a false classification from the minority and majority class. We can set ``class_weight="balanced"`` such that the weight applied is inversely proportional to the class frequency. We will test this parametrization in both linear model and tree-based model.


* Logistic Regression: ``LogisticRegression(class_weight='balanced')``
* Random Forest: ``RandomForestClassifier(class_weight='balanced')``

And there are more models that have this hyperparameter

* Decision Trees: ``DecisionTreeClassifier(class_weight='balanced')``
* Support Vector Machines: `SVC(class_weight='balanced')`
* Gradient Boosting: `GradientBoostingClassifier(class_weight='balanced')`

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score


log_reg = LogisticRegression(class_weight='balanced', random_state=42)
log_reg.fit(X_train, y_train)  #train the model

y_pred_logreg_classweight = log_reg.predict(X_test)  # Make predictions

# Evaluation metrics
acc_lg =  accuracy_score(y_test, y_pred_logreg_classweight)
balanced_acc_lg = balanced_accuracy_score(y_test, y_pred_logreg_classweight)
recall_lg = recall_score(y_test, y_pred_logreg_classweight, average='binary')
precision_lg = precision_score(y_test, y_pred_logreg_classweight, average='binary')


In [ ]:
# Save the results in the Dataframe

results_df.loc[len(results_df)] = [
    "Logistic Regression (Class weight)",
    acc_lg,
    balanced_acc_lg,
    recall_lg,
    precision_lg
]

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Random Forest
random_forest = RandomForestClassifier(class_weight='balanced', random_state=42)
random_forest.fit(X_train, y_train)

# Predictions
y_pred_forest_classweight = random_forest.predict(X_test)

# evaluation metrics
acc_rf = accuracy_score(y_test, y_pred_forest_classweight)
balanced_acc_rf = balanced_accuracy_score(y_test, y_pred_forest_classweight)
recall_rf = recall_score(y_test, y_pred_forest_classweight, average='binary')
precision_rf = precision_score(y_test, y_pred_forest_classweight, average='binary')

In [ ]:
# Save the results in the Dataframe

results_df.loc[len(results_df)] = [
    "Random Forest (Class weight)",
    acc_rf,
    balanced_acc_rf,
    recall_rf,
    precision_rf
]

In [ ]:
results_df



<div style="background-color:#ccffcc; padding:10px; border-radius:5px;">

### <span style="color:blue">Exercise</span>
What are the results if instead using X_train y_train, we use the oversampling training dataset? 
    </div>

#### Which option would you choose? Why?

## Matthews Correlation Coefficient (MCC)

The **Matthews Correlation Coefficient (MCC)** is a metric used to evaluate the quality of binary classifications, particularly useful in situations with imbalanced classes.

**Interpretation of MCC:**
* Range: The MCC value ranges from -1 to 1.
    * 1: Perfect classification.
    * 0: No better than random guessing.
    * -1: Completely wrong classification.
    
**Advantages of MCC**
* Balanced Assessment: It considers all elements of the confusion matrix, providing a more balanced evaluation than precision or recall alone.
* Useful for Imbalanced Data: It effectively evaluates models in datasets where one class significantly outweighs the other.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import matthews_corrcoef
import seaborn as sns

mcc_log_over = matthews_corrcoef(y_test, y_pred_logreg_over)
mcc_rf_under = matthews_corrcoef(y_test, y_pred_forest_under)


# Create a DataFrame with the results
mcc_data = {
    "Model": ["Logistic Regression", "Random Forest"],
    "MCC": [mcc_log_over, mcc_rf_under]
}

mcc_df = pd.DataFrame(mcc_data)  # Seaborn library expects a dataframe as input

print("Logistic Regression: ", mcc_log_over)
print("Random Forest: ", mcc_rf_under)



<div style="background-color:#ccffcc; padding:10px; border-radius:5px;">

### <span style="color:blue">Exercise</span>
What is the MCC score of the Random forest baseline scenario? And the Random Forest Oversampling one? 
    </div>

## Other option! Let's try moving the threshold for logistic regression

 ``predict_proba()`` returns the model’s estimated probability that each sample belongs to each class.
For a binary classifier like Logistic Regression, it returns two columns:

* Column 0 → Probability the sample belongs to class 0
* Column 1 → Probability the sample belongs to class 1 (the “positive” class)



In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression


# Initialize and fit the logistic regression model
log_reg = LogisticRegression(random_state=42)
log_reg.fit(X_train, y_train)
y_proba = log_reg.predict_proba(X_test)
y_proba

In [ ]:

# Get predicted probabilities
y_pred_positiveclass = log_reg.predict_proba(X_test)[:, 1]  # Probability for the positive class


In [ ]:
y_pred_positiveclass

Here, a **custom threshold of 0.2** is set. This means that any sample with a predicted probability of at least 0.2 will be classified as class 1, and samples with predicted probabilities below 0.2 will be classified as class 0.

`.astype(int)`:

The ``.astype(int)`` method converts the boolean array (``True/False``) into an integer array (``1/0``). In Python, ``True`` is equivalent to ``1`` and ``False``is equivalent to ``0``.

In [ ]:

# Set a custom threshold
threshold = 0.5  # Example threshold
y_pred_custom_threshold = (y_pred_positiveclass >= threshold).astype(int)

# Evaluation with custom threshold
print("Custom Threshold Logistic Regression Accuracy: ", accuracy_score(y_test, y_pred_custom_threshold))
print(classification_report(y_test, y_pred_custom_threshold))
print(confusion_matrix(y_test, y_pred_custom_threshold))

plot_confusion_matrix(y_test, y_pred_custom_threshold, "Logistic Regression")

In [ ]:
# evaluation metrics
acc_lg = accuracy_score(y_test, y_pred_custom_threshold)
balanced_acc_lg = balanced_accuracy_score(y_test, y_pred_custom_threshold)
recall_lg = recall_score(y_test, y_pred_custom_threshold, average='binary')
precision_lg = precision_score(y_test, y_pred_custom_threshold, average='binary')



In [ ]:
# Save the results in the Dataframe

results_df.loc[len(results_df)] = [
    "Logistic regression (threshold)",
    balanced_acc_lg,
    balanced_acc_rf,
    recall_lg,
    precision_lg
]

In [ ]:
results_df